# Mejoras — Opción A: Features de Tendencia

## Caso Práctico - Empresa de Telecomunicaciones
## Prácticas Aplicadas 2026

---

## Motivación

En los notebooks anteriores el mejor AUC obtenido fue **0.690** (LR GridSearch). El techo con las variables actuales parece estar ahí.

La hipótesis de este notebook es que **el churn no lo causa un evento puntual, sino un deterioro progresivo**. Un cliente que lleva 3 meses pagando tarde, cuyo consumo lleva bajando o cuya zona lleva meses con mala red, tiene más riesgo que uno que tuvo un mal mes puntual.

Para capturar eso añadimos **variables de tendencia**:
- **Tendencia de importe**: ¿está pagando menos que hace 3 meses?
- **Tendencia de stress de red**: ¿lleva meses con peor red?
- **Racha de impagos**: ¿cuántos meses consecutivos lleva con impago?
- **Satisfacción histórica**: media de satisfacción en las últimas interacciones con soporte
- **Flag de desengagement**: ¿lleva 2+ meses sin consumo extra?


## 1. Librerías


In [ ]:
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, average_precision_score,
    RocCurveDisplay, PrecisionRecallDisplay
)

try:
    from xgboost import XGBClassifier
    XGBOOST_OK = True
except ImportError:
    XGBOOST_OK = False
    print('XGBoost no instalado — se omitirá')

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
sns.set_theme(style='whitegrid', context='notebook')

ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

from src.load import load_all
from src.clean import clean_all

RANDOM_STATE = 42
print('Librerías cargadas')

## 2. Carga de datos


In [ ]:
data  = load_all()
clean = clean_all(data, save=False)

clientes  = clean['clientes']
churn     = clean['churn']
factura   = clean['facturacion']
soporte   = clean['soporte']
calidad   = clean['calidad']

print('\nDatos cargados')

---
## 3. Construcción del panel con features de tendencia

Mantenemos todas las variables del modelo anterior y añadimos las nuevas features de tendencia.


In [ ]:
# ── Facturación mensual base ───────────────────────────────────────────────
factura['mes'] = factura['fecha'].dt.to_period('M').dt.to_timestamp()

factura_mes = factura.groupby(['cliente_id', 'mes']).agg(
    importe_mes       = ('importe_total',        'sum'),
    impago_mes        = ('impago_flag',           'max'),
    dias_retraso_mes  = ('dias_retraso_pago',     'max'),
    stress_mes        = ('stress_calidad_lag',    'mean'),
    consumo_extra_mes = ('consumo_extra',         'sum'),
    variacion_consumo = ('variacion_consumo_pct', 'mean'),
    descuento_mes     = ('descuento_aplicado',    'sum'),
).reset_index().sort_values(['cliente_id', 'mes'])

print(f'Facturación mensual: {len(factura_mes):,} registros')

In [ ]:
# ── NUEVAS FEATURES DE TENDENCIA ──────────────────────────────────────────

# 1. Tendencia de importe: diferencia entre t-1 y t-3
#    Positivo = está pagando más que hace 3 meses (buena señal)
#    Negativo = está pagando menos (posible desengagement)
factura_mes['tendencia_importe'] = (
    factura_mes.groupby('cliente_id')['importe_mes']
    .transform(lambda x: x.shift(1) - x.shift(3))
)

# 2. Tendencia de stress de red: diferencia entre t-1 y t-3
#    Positivo = la red está empeorando en su zona
#    Negativo = la red está mejorando
factura_mes['tendencia_stress'] = (
    factura_mes.groupby('cliente_id')['stress_mes']
    .transform(lambda x: x.shift(1) - x.shift(3))
)

# 3. Racha de impagos consecutivos
#    Cuántos meses seguidos lleva con impago_flag=1
#    Un cliente con racha=3 es mucho más preocupante que uno con racha=1
impago_lag = factura_mes.groupby('cliente_id')['impago_mes'].shift(1).fillna(0)
grupos_racha = (impago_lag != impago_lag.groupby(factura_mes['cliente_id']).shift(1)).cumsum()
factura_mes['racha_impagos'] = (
    impago_lag * factura_mes.groupby(grupos_racha).cumcount().add(1)
)

# 4. Flag de desengagement: 2+ meses consecutivos sin consumo extra
#    Un cliente que ha dejado de consumir extras puede estar preparándose para irse
sin_consumo = (factura_mes['consumo_extra_mes'] <= 0).astype(int)
factura_mes['sin_consumo_lag1'] = factura_mes.groupby('cliente_id')[sin_consumo.name if hasattr(sin_consumo, 'name') else 'consumo_extra_mes'].transform(lambda x: (x.shift(1) <= 0).astype(int))
factura_mes['sin_consumo_2m'] = (
    factura_mes.groupby('cliente_id')['consumo_extra_mes']
    .transform(lambda x: ((x.shift(1) <= 0) & (x.shift(2) <= 0)).astype(int))
)

# 5. Rolling 3 meses (ya teníamos algunos, añadimos días de retraso)
for col in ['impago_mes', 'stress_mes', 'variacion_consumo', 'dias_retraso_mes']:
    factura_mes[f'{col}_roll3'] = (
        factura_mes.groupby('cliente_id')[col]
        .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    )

print('Features de tendencia creadas:')
print(f'  tendencia_importe  — nulos: {factura_mes["tendencia_importe"].isnull().sum():,}')
print(f'  tendencia_stress   — nulos: {factura_mes["tendencia_stress"].isnull().sum():,}')
print(f'  racha_impagos      — nulos: {factura_mes["racha_impagos"].isnull().sum():,}')
print(f'  sin_consumo_2m     — nulos: {factura_mes["sin_consumo_2m"].isnull().sum():,}')

In [ ]:
# ── Satisfacción histórica de soporte (últimos 6 meses con interacción) ───
# En lugar de usar la satisfacción del mes concreto (90% nulos),
# usamos la media histórica de los últimos 6 meses en que el cliente llamó

soporte['mes_dt'] = soporte['mes'].dt.to_period('M').dt.to_timestamp()

sop_hist = (soporte.groupby(['cliente_id', 'mes_dt'])
            ['satisfaccion_post'].mean()
            .reset_index()
            .sort_values(['cliente_id', 'mes_dt']))

sop_hist['satisfaccion_hist6'] = (
    sop_hist.groupby('cliente_id')['satisfaccion_post']
    .transform(lambda x: x.shift(1).rolling(6, min_periods=1).mean())
)
sop_hist = sop_hist.rename(columns={'mes_dt': 'mes'})

# Soporte mensual para lag 1
soporte_mes = soporte.groupby(['cliente_id', 'mes_dt']).agg(
    n_interacciones_mes = ('interaccion_id',    'count'),
    n_baja_mes          = ('motivo', lambda x: (x == 'Baja / portabilidad').sum()),
).reset_index().rename(columns={'mes_dt': 'mes'})

print(f'Satisfacción hist6 — clientes con historial: {sop_hist["cliente_id"].nunique():,}')
print(f'Nulos en satisfaccion_hist6: {sop_hist["satisfaccion_hist6"].isnull().sum():,}')

In [ ]:
# ── Construcción del panel ─────────────────────────────────────────────────
panel = churn.copy()
panel['mes'] = panel['fecha'].dt.to_period('M').dt.to_timestamp()

# Lag 1 — facturación (con features de tendencia incluidas)
cols_factura = [
    'importe_mes', 'impago_mes', 'dias_retraso_mes', 'stress_mes',
    'consumo_extra_mes', 'variacion_consumo',
    'tendencia_importe', 'tendencia_stress', 'racha_impagos', 'sin_consumo_2m',
    'impago_mes_roll3', 'stress_mes_roll3', 'variacion_consumo_roll3', 'dias_retraso_mes_roll3',
]
factura_lag = factura_mes[['cliente_id', 'mes'] + cols_factura].copy()
factura_lag['mes'] = factura_lag['mes'] + pd.DateOffset(months=1)
factura_lag = factura_lag.rename(columns={
    c: f'{c}_lag1' for c in cols_factura
})
panel = panel.merge(factura_lag, on=['cliente_id', 'mes'], how='left')

# Lag 1 — soporte
soporte_lag = soporte_mes.copy()
soporte_lag['mes'] = soporte_lag['mes'] + pd.DateOffset(months=1)
soporte_lag = soporte_lag.rename(columns={
    c: f'{c}_lag1' for c in ['n_interacciones_mes', 'n_baja_mes']
})
panel = panel.merge(soporte_lag, on=['cliente_id', 'mes'], how='left')
panel['n_interacciones_mes_lag1'] = panel['n_interacciones_mes_lag1'].fillna(0)
panel['n_baja_mes_lag1']          = panel['n_baja_mes_lag1'].fillna(0)

# Satisfacción histórica — lag 1
sat_lag = sop_hist[['cliente_id', 'mes', 'satisfaccion_hist6']].copy()
sat_lag['mes'] = sat_lag['mes'] + pd.DateOffset(months=1)
panel = panel.merge(sat_lag, on=['cliente_id', 'mes'], how='left')

# Calidad de red — lag 1
calidad['mes'] = calidad['fecha'].dt.to_period('M').dt.to_timestamp()
calidad_lag = calidad[['zona_id', 'mes', 'indice_calidad_global',
                         'tasa_cortes_pct', 'cobertura_5g_pct']].copy()
calidad_lag['mes'] = calidad_lag['mes'] + pd.DateOffset(months=1)
calidad_lag = calidad_lag.rename(columns={
    'indice_calidad_global': 'calidad_global_lag1',
    'tasa_cortes_pct':       'tasa_cortes_lag1',
    'cobertura_5g_pct':      'cobertura_5g_lag1',
})
zona_cliente = clientes[['cliente_id', 'zona_id']]
panel = panel.merge(zona_cliente, on='cliente_id', how='left')
panel = panel.merge(calidad_lag, on=['zona_id', 'mes'], how='left')

# Perfil estático
cols_perfil = ['cliente_id', 'edad', 'sexo', 'estado_civil', 'num_lineas',
               'tipo_plan', 'tipo_zona', 'region', 'ingreso_estimado',
               'antiguedad_meses', 'descuento_activo', 'tipo_dispositivo']
cols_perfil_disp = [c for c in cols_perfil if c in clientes.columns]
panel = panel.merge(clientes[cols_perfil_disp], on='cliente_id', how='left')
panel['tipo_plan_num'] = panel['tipo_plan'].map({'Básico': 1, 'Estándar': 2, 'Premium': 3})

print(f'Panel: {panel.shape[0]:,} filas x {panel.shape[1]} columnas')
print(f'Tasa de churn: {panel["churn"].mean()*100:.2f}%')

## 4. Selección de variables

Añadimos las nuevas features de tendencia a las variables del modelo anterior.


In [ ]:
FEATURES_NUM = [
    # Perfil
    'edad', 'num_lineas', 'ingreso_estimado', 'antiguedad_meses', 'descuento_activo',
    # Facturación lag 1
    'importe_mes_lag1', 'impago_mes_lag1', 'dias_retraso_mes_lag1',
    'stress_mes_lag1', 'consumo_extra_mes_lag1', 'variacion_consumo_lag1',
    # Rolling 3 meses
    'impago_mes_roll3_lag1', 'stress_mes_roll3_lag1',
    'variacion_consumo_roll3_lag1', 'dias_retraso_mes_roll3_lag1',
    # ── NUEVAS: tendencia ──
    'tendencia_importe_lag1',   # ¿está pagando menos que hace 3 meses?
    'tendencia_stress_lag1',    # ¿la red empeora en su zona?
    'racha_impagos_lag1',       # meses consecutivos con impago
    'sin_consumo_2m_lag1',      # 2+ meses sin consumo extra
    # ── NUEVAS: satisfacción histórica ──
    'satisfaccion_hist6',       # media satisfacción últimas 6 interacciones
    # Soporte lag 1
    'n_interacciones_mes_lag1', 'n_baja_mes_lag1',
    # Calidad de red lag 1
    'calidad_global_lag1', 'tasa_cortes_lag1', 'cobertura_5g_lag1',
]

FEATURES_CAT_NOMINAL = [
    'region', 'tipo_zona', 'sexo', 'estado_civil', 'tipo_dispositivo',
]

FEATURES_ORDINAL = ['tipo_plan_num']

TARGET = 'churn'
all_features = FEATURES_NUM + FEATURES_CAT_NOMINAL + FEATURES_ORDINAL

missing = [f for f in all_features if f not in panel.columns]
if missing:
    print(f'⚠️ No encontradas: {missing}')
else:
    print(f'✅ {len(all_features)} variables disponibles')
    print(f'   Nuevas respecto al modelo anterior: tendencia_importe, tendencia_stress,'
          f' racha_impagos, sin_consumo_2m, satisfaccion_hist6')

## 5. División train / test y preprocesado


In [ ]:
# Split por cliente (mismo criterio que modelos anteriores)
clientes_unicos = panel['cliente_id'].unique()
np.random.seed(RANDOM_STATE)
clientes_test  = np.random.choice(clientes_unicos,
                                   size=int(len(clientes_unicos) * 0.2),
                                   replace=False)
clientes_train = np.setdiff1d(clientes_unicos, clientes_test)

train = panel[panel['cliente_id'].isin(clientes_train)].dropna(
    subset=['importe_mes_lag1', 'stress_mes_lag1']).copy()
test  = panel[panel['cliente_id'].isin(clientes_test)].dropna(
    subset=['importe_mes_lag1', 'stress_mes_lag1']).copy()

X_train = train[all_features]
y_train = train[TARGET]
X_test  = test[all_features]
y_test  = test[TARGET]

RATIO = round((y_train == 0).sum() / y_train.sum())

print(f'Train: {len(X_train):,} filas | {y_train.mean()*100:.2f}% churn')
print(f'Test:  {len(X_test):,} filas  | {y_test.mean()*100:.2f}% churn')
print(f'Ratio desbalance: {RATIO}:1')

# Preprocesador
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
nom_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='desconocido')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
])
ord_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, FEATURES_NUM),
    ('nom', nom_pipeline, FEATURES_CAT_NOMINAL),
    ('ord', ord_pipeline, FEATURES_ORDINAL),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print('\nPreprocesador definido')

## 6. Función de evaluación


In [ ]:
def evaluar_modelo(nombre, y_true, y_pred_proba, umbral=0.05):
    y_pred = (y_pred_proba >= umbral).astype(int)
    return {
        'Modelo':    nombre,
        'AUC-ROC':   round(roc_auc_score(y_true, y_pred_proba), 3),
        'AUC-PR':    round(average_precision_score(y_true, y_pred_proba), 3),
        'Accuracy':  round(accuracy_score(y_true, y_pred), 3),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 3),
        'Recall':    round(recall_score(y_true, y_pred, zero_division=0), 3),
        'F1':        round(f1_score(y_true, y_pred, zero_division=0), 3),
    }

resultados = []
probas     = {}

# Referencia del mejor modelo anterior
resultados.append({
    'Modelo': 'LR GridSearch (anterior)',
    'AUC-ROC': 0.690, 'AUC-PR': 0.016,
    'Accuracy': None, 'Precision': None, 'Recall': None, 'F1': None
})

print('Función de evaluación definida')

---
## 7. GridSearch — Los tres modelos con features de tendencia

Misma estructura de diccionario que en el notebook anterior. Comparamos si las nuevas features mejoran el AUC.


In [ ]:
# Preprocesamos para XGBoost
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

model_configs = {
    'Regresión Logística': {
        'estimator': Pipeline([
            ('preprocessor', preprocessor),
            ('model', LogisticRegression(
                class_weight='balanced',
                random_state=RANDOM_STATE,
                max_iter=1000, solver='saga'
            ))
        ]),
        'params':   {'model__C': [0.01, 0.1, 1.0], 'model__penalty': ['l1', 'l2']},
        'usa_prep': False,
    },
    'Random Forest': {
        'estimator': Pipeline([
            ('preprocessor', preprocessor),
            ('model', RandomForestClassifier(
                class_weight='balanced',
                random_state=RANDOM_STATE, n_jobs=-1
            ))
        ]),
        'params':   {
            'model__n_estimators':     [100, 200],
            'model__max_depth':        [5, 8],
            'model__min_samples_leaf': [20, 50],
        },
        'usa_prep': False,
    },
}

if XGBOOST_OK:
    model_configs['XGBoost'] = {
        'estimator': XGBClassifier(
            scale_pos_weight=RATIO,
            random_state=RANDOM_STATE,
            eval_metric='auc', verbosity=0
        ),
        'params': {
            'n_estimators':  [200, 400],
            'max_depth':     [4, 6],
            'learning_rate': [0.01, 0.05],
            'subsample':     [0.7, 0.9],
        },
        'usa_prep': True,
    }

best_estimators = {}

for nombre, config in model_configs.items():
    print(f'\n--- GridSearch: {nombre} ---')
    X_tr = X_train_prep if config['usa_prep'] else X_train
    X_te = X_test_prep  if config['usa_prep'] else X_test

    gs = GridSearchCV(
        config['estimator'], config['params'],
        cv=cv, scoring='roc_auc', n_jobs=-1, verbose=1
    )
    gs.fit(X_tr, y_train)

    print(f'  Mejores parámetros: {gs.best_params_}')
    print(f'  Mejor AUC en CV:    {gs.best_score_:.3f}')

    proba = gs.predict_proba(X_te)[:, 1]
    probas[nombre] = proba
    best_estimators[nombre] = gs.best_estimator_

    res = evaluar_modelo(nombre, y_test, proba)
    resultados.append(res)
    print(f'  AUC test: {res["AUC-ROC"]:.3f} | Recall: {res["Recall"]:.3f} | F1: {res["F1"]:.3f}')

print('\n✅ GridSearch completado')

---
## 8. Comparativa: ¿mejoran las features de tendencia?


In [ ]:
df_res = pd.DataFrame(resultados).set_index('Modelo')
print('=== COMPARATIVA ===')
display(df_res.style
        .highlight_max(subset=['AUC-ROC','AUC-PR','Recall','F1'], color='#c8f5c8')
        .format('{:.3f}', na_rep='-'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curvas ROC
colores = {'Regresión Logística': '#4C9BE8', 'Random Forest': '#f39c12', 'XGBoost': '#E85C4C'}
for nombre, proba in probas.items():
    RocCurveDisplay.from_predictions(
        y_test, proba, name=nombre,
        color=colores.get(nombre, 'gray'), ax=axes[0]
    )
axes[0].plot([0,1],[0,1],'k--', label='Baseline (0.500)')
axes[0].axvline(x=0, ymin=0, ymax=0, color='purple', linestyle='-',
                label=f'Anterior LR GS (AUC=0.690)')
axes[0].set_title('Curvas ROC — Opción A (features de tendencia)', fontweight='bold')
axes[0].legend(fontsize=8)

# Curvas PR
for nombre, proba in probas.items():
    PrecisionRecallDisplay.from_predictions(
        y_test, proba, name=nombre,
        color=colores.get(nombre, 'gray'), ax=axes[1]
    )
axes[1].set_title('Curvas Precision-Recall', fontweight='bold')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Importancia de variables — LR (coeficientes) y RF (feature importance)
if 'Regresión Logística' in best_estimators:
    lr = best_estimators['Regresión Logística']
    nom_names = (lr.named_steps['preprocessor']
                 .named_transformers_['nom']
                 .named_steps['encoder']
                 .get_feature_names_out(FEATURES_CAT_NOMINAL))
    feat_names = FEATURES_NUM + list(nom_names) + FEATURES_ORDINAL
    coefs = pd.Series(lr.named_steps['model'].coef_[0], index=feat_names)
    top20 = coefs.sort_values(key=abs, ascending=False).head(20).sort_values()

    fig, ax = plt.subplots(figsize=(9, 7))
    colors = ['#E85C4C' if v > 0 else '#4C9BE8' for v in top20.values]
    ax.barh(top20.index, top20.values, color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title('LR Opción A — Top 20 variables\n'
                 '(rojo = aumenta churn | azul = lo reduce)',
                 fontweight='bold')
    ax.set_xlabel('Coeficiente (escala estandarizada)')

    # Marcar las variables nuevas
    nuevas = ['tendencia_importe_lag1', 'tendencia_stress_lag1',
              'racha_impagos_lag1', 'sin_consumo_2m_lag1', 'satisfaccion_hist6']
    for label in ax.get_yticklabels():
        if label.get_text() in nuevas:
            label.set_color('#9b59b6')
            label.set_fontweight('bold')

    ax.text(0.98, 0.02, '* Variables nuevas en morado',
            transform=ax.transAxes, ha='right', fontsize=8,
            color='#9b59b6')
    plt.tight_layout()
    plt.show()

---
## 9. Conclusiones

### ¿Mejoran las features de tendencia el AUC?

**Sí, y de forma clara.** La Regresión Logística con features de tendencia alcanza un AUC de **0.701**, subiendo desde el 0.690 del mejor modelo anterior. Es la primera vez en todo el proyecto que superamos el 0.70 con un modelo sin leakage.

| Modelo | AUC-ROC | AUC-PR | AUC en CV |
|--------|---------|--------|----------|
| LR GridSearch (anterior) | 0.690 | 0.016 | 0.705 |
| **Regresión Logística (Opción A)** | **0.701** | **0.018** | **0.714** |
| XGBoost (Opción A) | 0.698 | 0.018 | 0.703 |
| Random Forest (Opción A) | 0.692 | 0.017 | 0.703 |

El AUC en validación cruzada de la LR sube a **0.714**, lo que confirma que la mejora es robusta y no es un artefacto del conjunto de test.

### ¿Qué aportan las variables nuevas?

Mirando el gráfico de coeficientes, dos de las cinco variables nuevas aparecen en el top 20:

- **`racha_impagos_lag1`** — es la **3ª variable más influyente** en aumentar el churn, solo por detrás de `impago_mes_lag1` y `tipo_zona_urbana_premium`. Esto confirma la hipótesis: un cliente que lleva varios meses consecutivos sin pagar tiene un riesgo muy superior a uno que tuvo un impago puntual. La tendencia importa, no solo el dato mensual.

- **`sin_consumo_2m_lag1`** — también aparece en el top 20 con coeficiente positivo. Llevar 2 meses seguidos sin consumo extra es una señal de desengagement que el modelo está captando.

Las otras tres variables nuevas (`tendencia_importe`, `tendencia_stress`, `satisfaccion_hist6`) no aparecen en el top 20. `tendencia_importe` y `tendencia_stress` tienen muchos nulos (29.915 de 32.141 registros) porque necesitan 3 meses de historial, lo que limita su señal. `satisfaccion_hist6` solo está disponible para los 9.973 clientes que alguna vez llamaron a soporte.

### ¿Por qué `calidad_global_lag1` tiene coeficiente negativo?

Aparece como la variable que más **reduce** el churn, lo que es contraintuitivo (más calidad global = menos churn tiene sentido, pero el signo negativo en la LR puede deberse a multicolinealidad con otras variables de red como `cobertura_5g_lag1` y `stress_mes_lag1`). Es una limitación conocida de la regresión logística cuando hay variables correlacionadas entre sí.

### Resumen global del proyecto

| Modelo | AUC | Features tendencia | Leakage | Producción |
|--------|-----|--------------------|---------|------------|
| Binario LR | 0.991 | No | Sí | No |
| Binario RF | 0.994 | No | Sí | No |
| Temporal LR | 0.685 | No | No | Sí |
| LR GridSearch | 0.690 | No | No | Sí |
| **LR Opción A (modelo final)** | **0.701** | **Sí** | **No** | **Sí** |

### Modelo final recomendado

**Regresión Logística con features de tendencia** (`C=0.1`, `penalty='l1'`, `class_weight='balanced'`):
- AUC = 0.701 — mejor resultado sin leakage del proyecto
- Interpretable: los coeficientes explican directamente qué variables suben o bajan el riesgo
- Las variables más importantes son coherentes con el negocio: impago, racha de impagos, stress de red, antigüedad
- Genera cada mes una lista priorizada de clientes en riesgo

### Resumen hasta ahora

Con AUC = 0.701 ya tenemos un resultado sólido y explicable. La narrativa es clara:
1. Empezamos con un modelo binario que daba 0.99 pero tenía leakage
2. El modelo temporal correcto daba 0.685
3. Con ingeniería de features (variables de tendencia) subimos a 0.701
4. El valor del modelo no está en el AUC exacto sino en identificar qué clientes tienen más riesgo para priorizar las acciones de retención

---
*Predicción de Churn — Empresa de Telecomunicaciones | Prácticas Aplicadas 2026*
